In [31]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import ExtraTreesRegressor
from IPython.display import display

# 1. LOAD DATA & PREPROCESSING DASAR

In [32]:
df = pd.read_csv('sensor_log_20260928_0925.csv')

# Ekstrak Jam (hour) dan Menit (minute) dari kolom 'waktu' (Waktu Lokal)

In [33]:
df['waktu_dt'] = pd.to_datetime(df['waktu'], format='%H:%M:%S', errors='coerce')
df['hour'] = df['waktu_dt'].dt.hour
df['minute'] = df['waktu_dt'].dt.minute

# Filter data bersih dari anomali sensor (menghilangkan nilai temp/hum <= 0)

In [34]:
data_clean = df[(df['temp'] > 0) & (df['hum'] > 0)][['temp', 'hum', 'lux', 'hour', 'minute']].copy()

# 2. TRAIN MODEL IMPUTER (MICE)

In [35]:
imputer = IterativeImputer(
    estimator=ExtraTreesRegressor(n_estimators=50, random_state=42),
    max_iter=10,
    random_state=42
)
imputer.fit(data_clean)

,"estimator estimator: estimator object, default=BayesianRidge()The estimator to use at each step of the round-robin imputation.If `sample_posterior=True`, the estimator must support`return_std` in its `predict` method.",ExtraTreesReg...ndom_state=42)
,"random_state random_state: int, RandomState instance or None, default=NoneThe seed of the pseudo random number generator to use. Randomizesselection of estimator features if `n_nearest_features` is not `None`,the `imputation_order` if `random`, and the sampling from posterior if`sample_posterior=True`. Use an integer for determinism.See :term:`the Glossary <random_state>`.",42
,"missing_values missing_values: int or np.nan, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`should be set to `np.nan`, since `pd.NA` will be converted to `np.nan`.",nan
,"sample_posterior sample_posterior: bool, default=FalseWhether to sample from the (Gaussian) predictive posterior of thefitted estimator for each imputation. Estimator must support`return_std` in its `predict` method if set to `True`. Set to`True` if using `IterativeImputer` for multiple imputations.",False
,"max_iter max_iter: int, default=10Maximum number of imputation rounds to perform before returning theimputations computed during the final round. A round is a singleimputation of each feature with missing values. The stopping criterionis met once `max(abs(X_t - X_{t-1}))/max(abs(X[known_vals])) < tol`,where `X_t` is `X` at iteration `t`. Note that early stopping is onlyapplied if `sample_posterior=False`.",10
,"tol tol: float, default=1e-3Tolerance of the stopping condition.",0.001
,"n_nearest_features n_nearest_features: int, default=NoneNumber of other features to use to estimate the missing values ofeach feature column. Nearness between features is measured usingthe absolute correlation coefficient between each feature pair (afterinitial imputation). To ensure coverage of features throughout theimputation process, the neighbor features are not necessarily nearest,but are drawn with probability proportional to correlation for eachimputed target feature. Can provide significant speed-up when thenumber of features is huge. If `None`, all features will be used.",None
,"initial_strategy initial_strategy: {'mean', 'median', 'most_frequent', 'constant'}, default='mean'Which strategy to use to initialize the missing values. Same as the`strategy` parameter in :class:`~sklearn.impute.SimpleImputer`.",'mean'
,"fill_value fill_value: str or numerical value, default=NoneWhen `strategy=""constant""`, `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types... versionadded:: 1.3",None
,"imputation_order imputation_order: {'ascending', 'descending', 'roman', 'arabic', 'random'}, default='ascending'The order in which the features will be imputed. Possible values:- `'ascending'`: From features with fewest missing values to most.- `'descending'`: From features with most missing values to fewest.- `'roman'`: Left to right.- `'arabic'`: Right to left.- `'random'`: A random order for each round.",'ascending'
,"skip_complete skip_complete: bool, default=FalseIf `True` then features with missing values during :meth:`transform`which did not have any missing values during :meth:`fit` will beimputed with the initial imputation method only. Set to `True` if youhave many features with no missing values at both :meth:`fit` and:meth:`transform` time to save compute.",False


# 3. FUNGSI PREDIKSI & VALIDASI DATA

In [54]:
def validasi_prediksi_range(hour, minute, temp=None, hum=None, lux=None):
    # Prepared DataFrame Input
    input_df = pd.DataFrame([[temp, hum, lux, hour, minute]], columns=['temp', 'hum', 'lux', 'hour', 'minute'])
    
    # Prediksi menggunakan imputer
    pred_res = imputer.transform(input_df)
    pred_temp, pred_hum, pred_lux, _, _ = pred_res[0]
    
    # Cari Data Asli
    data_asli = df[(df['hour'] == hour) & (df['minute'] == minute) & (df['temp'] > 0) & (df['hum'] > 0)]
    
    print(f"=== HASIL EVALUASI MODEL PUKUL {hour:02d}:{minute:02d} ===\n")
    
    if not data_asli.empty:
        # Menghitung Rata-rata Asli
        mean_temp = data_asli['temp'].mean()
        mean_hum  = data_asli['hum'].mean()
        mean_lux  = data_asli['lux'].mean()
        
        # Fungsi kecil untuk menghitung persentase capaian (menghindari error jika rata-rata 0)
        def hitung_capaian(pred, mean):
            if mean == 0:
                return "100.00 %" if pred == 0 else "0.00 %"
            
            # Rumus: 100 - (|Pred - Mean| / Mean * 100)
            error_persen = abs(pred - mean) / mean * 100
            capaian = 100 - error_persen
            return f"{max(0, capaian):.2f} %" # max(0) agar tidak minus jika prediksi ngaco
        
        # Hitung Capaian
        capaian_temp = hitung_capaian(pred_temp, mean_temp)
        capaian_hum  = hitung_capaian(pred_hum, mean_hum)
        capaian_lux  = hitung_capaian(pred_lux, mean_lux)
        
        # ==========================================
        # 1. TABEL RINGKASAN
        # ==========================================
        summary_data = {
            'Keterangan': [
                '1. PREDIKSI', 
                '2. RATA-RATA', 
                '3. CAPAIAN (Akurasi)', 
                '4. MAX (Tertinggi)', 
                '5. MIN (Terendah)'
            ],
            'temp': [
                round(pred_temp, 2), 
                round(mean_temp, 2), 
                capaian_temp, 
                round(data_asli['temp'].max(), 2), 
                round(data_asli['temp'].min(), 2)
            ],
            'hum': [
                round(pred_hum, 2), 
                round(mean_hum, 2), 
                capaian_hum, 
                round(data_asli['hum'].max(), 2), 
                round(data_asli['hum'].min(), 2)
            ],
            'lux': [
                round(pred_lux, 2), 
                round(mean_lux, 2), 
                capaian_lux, 
                round(data_asli['lux'].max(), 2), 
                round(data_asli['lux'].min(), 2)
            ]
        }
        df_summary = pd.DataFrame(summary_data)
        
        # ==========================================
        # 2. TABEL KESELURUHAN DATA ASLI
        # ==========================================
        kolom_tampil = []
        if 'tanggal' in data_asli.columns: kolom_tampil.append('tanggal')
        if 'waktu' in data_asli.columns: kolom_tampil.append('waktu')
        kolom_tampil.extend(['temp', 'hum', 'lux'])
        
        df_raw = data_asli[kolom_tampil].copy()
        
        print("▶ TABEL 1: RINGKASAN PREDIKSI & RANGE AKTUAL")
        display(df_summary)
        
        print("\n▶ TABEL 2: KESELURUHAN DATA ASLI (Referensinya)")
        display(df_raw)
        
        return df_summary, df_raw

    else:
        print("Data asli tidak ditemukan pada jam dan menit tersebut.")
        return None, None

# Testing

In [55]:
validasi_prediksi_range(hour=15, minute=30, hum=62.5, lux=150.0)

=== HASIL EVALUASI MODEL PUKUL 15:30 ===

▶ TABEL 1: RINGKASAN PREDIKSI & RANGE AKTUAL


,Keterangan,temp,hum,lux
0,1. PREDIKSI,28.23,62.5,150.0
1,2. RATA-RATA,34.04,35.27,1025.87
2,3. CAPAIAN (Akurasi),82.94 %,22.79 %,14.62 %
3,4. MAX (Tertinggi),38.5,73.0,2286.67
4,5. MIN (Terendah),27.6,15.0,107.5



▶ TABEL 2: KESELURUHAN DATA ASLI (Referensinya)


,tanggal,waktu,temp,hum,lux
317,2026-09-27,15:30:37,29.8,52.0,445.83
1558,2026-09-26,15:30:50,31.3,43.0,336.67
2661,2026-09-25,15:30:57,31.8,41.0,448.33
3605,2026-09-24,15:30:13,35.6,33.0,1578.33
4876,2026-09-23,15:30:24,35.2,33.0,1215.83
...,...,...,...,...,...
78970,27/07/2026,15:30:22,31.3,34.0,311.67
80383,26/07/2026,15:30:35,38.0,15.0,2007.50
81796,25/07/2026,15:30:48,38.0,24.0,2286.67
83209,24/07/2026,15:30:59,34.7,30.0,912.50


(             Keterangan     temp      hum      lux
 0           1. PREDIKSI    28.23     62.5    150.0
 1          2. RATA-RATA    34.04    35.27  1025.87
 2  3. CAPAIAN (Akurasi)  82.94 %  22.79 %  14.62 %
 3    4. MAX (Tertinggi)     38.5     73.0  2286.67
 4     5. MIN (Terendah)     27.6     15.0    107.5,
           tanggal     waktu  temp   hum      lux
 317    2026-09-27  15:30:37  29.8  52.0   445.83
 1558   2026-09-26  15:30:50  31.3  43.0   336.67
 2661   2026-09-25  15:30:57  31.8  41.0   448.33
 3605   2026-09-24  15:30:13  35.6  33.0  1578.33
 4876   2026-09-23  15:30:24  35.2  33.0  1215.83
 ...           ...       ...   ...   ...      ...
 78970  27/07/2026  15:30:22  31.3  34.0   311.67
 80383  26/07/2026  15:30:35  38.0  15.0  2007.50
 81796  25/07/2026  15:30:48  38.0  24.0  2286.67
 83209  24/07/2026  15:30:59  34.7  30.0   912.50
 84619  23/07/2026  15:30:08  36.3  33.0  1950.83
 
 [63 rows x 5 columns])

In [56]:
validasi_prediksi_range(hour=8, minute=15, temp=25.1, lux=110.0)

=== HASIL EVALUASI MODEL PUKUL 08:15 ===

▶ TABEL 1: RINGKASAN PREDIKSI & RANGE AKTUAL


,Keterangan,temp,hum,lux
0,1. PREDIKSI,25.1,54.62,110.0
1,2. RATA-RATA,25.11,60.23,400.73
2,3. CAPAIAN (Akurasi),99.95 %,90.69 %,27.45 %
3,4. MAX (Tertinggi),27.6,79.0,730.83
4,5. MIN (Terendah),21.8,47.0,229.17



▶ TABEL 2: KESELURUHAN DATA ASLI (Referensinya)


,tanggal,waktu,temp,hum,lux
739,2026-09-27,08:15:20,24.8,79.0,582.50
3086,2026-09-25,08:15:41,26.2,59.0,440.83
3894,2026-09-24,08:15:57,25.3,57.0,349.17
5286,2026-09-23,08:15:06,25.3,66.0,462.50
6997,2026-09-20,08:15:49,25.3,62.0,387.50
8400,2026-09-19,08:15:57,26.2,77.0,453.33
9784,2026-09-18,08:15:07,27.1,53.0,535.00
11181,2026-09-17,08:15:22,25.8,50.0,364.17
12567,2026-09-16,08:15:41,25.3,49.0,301.67
13964,2026-09-15,08:15:58,24.1,48.0,229.17


(             Keterangan     temp      hum      lux
 0           1. PREDIKSI     25.1    54.62    110.0
 1          2. RATA-RATA    25.11    60.23   400.73
 2  3. CAPAIAN (Akurasi)  99.95 %  90.69 %  27.45 %
 3    4. MAX (Tertinggi)     27.6     79.0   730.83
 4     5. MIN (Terendah)     21.8     47.0   229.17,
           tanggal     waktu  temp   hum     lux
 739    2026-09-27  08:15:20  24.8  79.0  582.50
 3086   2026-09-25  08:15:41  26.2  59.0  440.83
 3894   2026-09-24  08:15:57  25.3  57.0  349.17
 5286   2026-09-23  08:15:06  25.3  66.0  462.50
 6997   2026-09-20  08:15:49  25.3  62.0  387.50
 8400   2026-09-19  08:15:57  26.2  77.0  453.33
 9784   2026-09-18  08:15:07  27.1  53.0  535.00
 11181  2026-09-17  08:15:22  25.8  50.0  364.17
 12567  2026-09-16  08:15:41  25.3  49.0  301.67
 13964  2026-09-15  08:15:58  24.1  48.0  229.17
 15374  2026-09-14  08:15:15  25.3  54.0  271.67
 16750  2026-09-13  08:15:29  26.7  57.0  584.17
 18153  2026-09-12  08:15:43  26.7  58.0  433.33
 

In [57]:
validasi_prediksi_range(hour=19, minute=20, temp=24.5)

=== HASIL EVALUASI MODEL PUKUL 19:20 ===

▶ TABEL 1: RINGKASAN PREDIKSI & RANGE AKTUAL


,Keterangan,temp,hum,lux
0,1. PREDIKSI,24.5,55.76,2.73
1,2. RATA-RATA,27.24,54.97,24.77
2,3. CAPAIAN (Akurasi),89.94 %,98.56 %,11.03 %
3,4. MAX (Tertinggi),31.3,91.0,1406.67
4,5. MIN (Terendah),24.1,32.0,0.83



▶ TABEL 2: KESELURUHAN DATA ASLI (Referensinya)


,tanggal,waktu,temp,hum,lux
1485,2026-09-26,19:20:03,24.1,91.0,0.83
2436,2026-09-25,19:20:10,28.0,49.0,0.83
4654,2026-09-23,19:20:41,28.5,47.0,0.83
6043,2026-09-22,19:20:52,25.8,78.0,0.83
7757,2026-09-19,19:20:31,25.3,69.0,0.83
9151,2026-09-18,19:20:36,24.8,78.0,0.83
10540,2026-09-17,19:20:54,27.6,52.0,0.83
11938,2026-09-16,19:20:08,27.6,48.0,0.83
13323,2026-09-15,19:20:28,28.0,32.0,0.83
14723,2026-09-14,19:20:47,28.9,36.0,0.83


(             Keterangan     temp      hum      lux
 0           1. PREDIKSI     24.5    55.76     2.73
 1          2. RATA-RATA    27.24    54.97    24.77
 2  3. CAPAIAN (Akurasi)  89.94 %  98.56 %  11.03 %
 3    4. MAX (Tertinggi)     31.3     91.0  1406.67
 4     5. MIN (Terendah)     24.1     32.0     0.83,
           tanggal     waktu  temp   hum      lux
 1485   2026-09-26  19:20:03  24.1  91.0     0.83
 2436   2026-09-25  19:20:10  28.0  49.0     0.83
 4654   2026-09-23  19:20:41  28.5  47.0     0.83
 6043   2026-09-22  19:20:52  25.8  78.0     0.83
 7757   2026-09-19  19:20:31  25.3  69.0     0.83
 9151   2026-09-18  19:20:36  24.8  78.0     0.83
 10540  2026-09-17  19:20:54  27.6  52.0     0.83
 11938  2026-09-16  19:20:08  27.6  48.0     0.83
 13323  2026-09-15  19:20:28  28.0  32.0     0.83
 14723  2026-09-14  19:20:47  28.9  36.0     0.83
 16122  2026-09-13  19:20:01  28.0  37.0     0.83
 17510  2026-09-12  19:20:13  27.6  58.0     0.83
 18873  2026-09-11  19:20:29  28.5  5